# MAX Engine Session
> Singleton session manager, graph caching, and core CPU/GPU compute ops (sum, mean, min, max, masked_sum).

In [ ]:
#| default_exp max_engine

In [ ]:
#| export
from pathlib import Path
import numpy as np
from typing import Optional, Tuple, Dict, Any, Literal, Union, Callable

from max import engine, driver
from max.graph import Graph, TensorType, ops, DeviceRef
from max.dtype import DType
from max.experimental import tensor as mx_tensor

DeviceType = Literal["cpu", "gpu"]
Scalar = Union[int, float]

## MAXSession
Singleton per device — compiles and caches MAX graphs keyed by `(op, shape, dtype)`.

In [ ]:
#| export
class MAXSession:
    """MAX Engine inference session for CPU or GPU, with compiled graph caching."""
    _instances: Dict[str, 'MAXSession'] = {}

    def __init__(self, device_type: DeviceType = "cpu"):
        self.device_type = device_type
        if device_type == "gpu":
            if driver.accelerator_count() == 0:
                raise RuntimeError("No GPU available. Use device='cpu'.")
            self.device     = driver.Accelerator()
            self.device_ref = DeviceRef.GPU()
        else:
            self.device     = driver.CPU()
            self.device_ref = DeviceRef.CPU()
        self.session = engine.InferenceSession(devices=[self.device])
        self._cache: Dict[Tuple[str, tuple, str], Any] = {}

    @classmethod
    def get(cls, device: DeviceType = "cpu") -> 'MAXSession':
        """Get or create the singleton session for `device`."""
        if device not in cls._instances:
            cls._instances[device] = cls(device)
        return cls._instances[device]

    @classmethod
    def reset(cls) -> None:
        """Destroy all sessions (useful in tests)."""
        cls._instances = {}

    def _get_or_compile(self, key: tuple, build_fn: Callable) -> Any:
        """Return cached compiled model or build+cache a new one."""
        if key not in self._cache:
            self._cache[key] = self.session.load(build_fn())
        return self._cache[key]

    def to_tensor(self, arr: np.ndarray) -> driver.Tensor:
        """numpy → contiguous float32 → MAX driver.Tensor on this session's device."""
        arr = np.ascontiguousarray(arr, dtype=np.float32)
        t   = driver.Tensor.from_numpy(arr)
        return t.to(self.device) if self.device_type == "gpu" else t

    def from_tensor(self, t: driver.Tensor) -> np.ndarray:
        """MAX driver.Tensor → numpy (D2H copy if on GPU)."""
        if self.device_type == "gpu":
            t = t.to(driver.CPU())
        return t.to_numpy()

## Experimental tensor helpers
For ops that have a direct API on `max.experimental.tensor` (mean, min, max, std).

In [ ]:
#| export
def _to_mx(arr: np.ndarray, device: DeviceType = "cpu") -> mx_tensor.Tensor:
    """numpy → MAX experimental tensor on the requested device."""
    arr = np.ascontiguousarray(arr, dtype=np.float32)
    t   = mx_tensor.Tensor.from_dlpack(arr)
    if device == "gpu":
        if driver.accelerator_count() == 0:
            raise RuntimeError("No GPU available. Use device='cpu'.")
        t = t.to(driver.Accelerator())
    return t


def _item(t: mx_tensor.Tensor) -> float:
    """Extract Python scalar from a scalar MAX tensor."""
    return float(np.from_dlpack(t.to(driver.CPU())))

## Core aggregation functions

In [ ]:
#| export
def max_sum(arr: np.ndarray, device: DeviceType = "cpu") -> float:
    """
    Reduce-sum using MAX Engine (Graph API + compiled session cache).

    Same compiled graph is reused for identical `(shape, dtype)` pairs.
    """
    sess   = MAXSession.get(device)
    tensor = sess.to_tensor(arr)
    key    = ("sum", tuple(tensor.shape), "float32")

    def build():
        with Graph("sum", input_types=[TensorType(DType.float32, tensor.shape, sess.device_ref)]) as g:
            g.output(ops.sum(g.inputs[0]))
        return g

    model  = sess._get_or_compile(key, build)
    result = model.execute(tensor)[0]
    return float(sess.from_tensor(result).flat[0])


def max_mean(arr: np.ndarray, device: DeviceType = "cpu") -> float:
    """Mean via MAX experimental tensor API."""
    return _item(_to_mx(arr, device).mean(axis=None))


def max_min(arr: np.ndarray, device: DeviceType = "cpu") -> float:
    """Element-wise min via MAX experimental tensor API."""
    t = _to_mx(arr, device)
    if hasattr(t, "min"):
        return _item(t.min(axis=None))
    return float(np.min(arr))  # fallback if API not yet stable


def max_max(arr: np.ndarray, device: DeviceType = "cpu") -> float:
    """Element-wise max via MAX experimental tensor API."""
    t = _to_mx(arr, device)
    if hasattr(t, "max"):
        return _item(t.max(axis=None))
    return float(np.max(arr))  # fallback


def max_count(arr: np.ndarray) -> int:
    """Element count (null-free arrays only)."""
    return int(arr.size)

## Masked sum (core GroupBy primitive)

In [ ]:
#| export
def max_masked_sum(
    mask:   np.ndarray,   # Boolean or float32 mask (1.0 = include)
    values: np.ndarray,   # Values to sum where mask is true
    device: DeviceType = "cpu",
) -> float:
    """
    `sum(mask * values)` compiled once per shape via MAX Graph API.

    This is the fundamental primitive for masked group aggregation on both
    CPU and GPU — the same compiled graph executes on whichever device the
    session owns.
    """
    sess = MAXSession.get(device)
    m_f  = np.ascontiguousarray(mask,   dtype=np.float32)
    v_f  = np.ascontiguousarray(values, dtype=np.float32)
    m_t  = sess.to_tensor(m_f)
    v_t  = sess.to_tensor(v_f)
    key  = ("masked_sum", tuple(m_t.shape), "float32")

    def build():
        tt = TensorType(DType.float32, m_t.shape, sess.device_ref)
        with Graph("masked_sum", input_types=[tt, tt]) as g:
            m, v = g.inputs
            g.output(ops.sum(ops.mul(m, v)))
        return g

    model  = sess._get_or_compile(key, build)
    result = model.execute(m_t, v_t)[0]
    return float(sess.from_tensor(result).flat[0])

## 🚀 GPU Dispatch via Custom Mojo Kernels

Up to here every operation uses MAX's **built-in Graph ops** (`ops.sum`, `ops.mul`, etc.).
Those are correct and cache-friendly for CPU, but on GPU each `ops.sum` inside a loop is a
**separate kernel launch** — which means data travels to VRAM and back multiple times.

### The faster path: custom Mojo kernels

A Mojo kernel runs **inside** the GPU thread hierarchy.  A single kernel launch processes
the entire array in one pass using **warp-level reduction** (32 threads → 1 partial sum).

```
Array (N elements)
  └─ 256 threads/block × ceil(N/256) blocks
       └─ Each warp (32 threads) reduces → 1 partial sum
            └─ ops.sum(partial_sums) → scalar   (one final tiny reduction)
```

**Architecture choice**: kernels are shipped as a pre-compiled `kernels.mojopkg` binary inside
the package.  At graph-build time MAX JIT-links it in via `custom_extensions=[KERNELS_PATH]`.
No Mojo compiler dependency at user install time.

**Fallback**: if `kernels.mojopkg` is missing (dev environment, incomplete install),
all functions transparently fall back to the pure-Graph path. CPU users are never broken.

In [ ]:
#| export
# ── Kernel package location ──────────────────────────────────────────────────
# `kernels.mojopkg` is compiled from archive/kernels/ and shipped inside the
# mxframe package.  Build it with:
#   mojo package archive/kernels -o mxframe/kernels.mojopkg
KERNELS_PATH = Path(__file__).parent / "kernels.mojopkg"


def kernels_available() -> bool:
    """Return True if the compiled Mojo kernel package exists on disk."""
    return KERNELS_PATH.exists()


# Output size helpers (shapes are baked into the Graph at compile time)
def _partial_sum_size(n: int) -> int:
    """Number of warp-partial-sum outputs for an array of length n."""
    from math import ceil
    WARP_SIZE = 32
    return ceil(n / WARP_SIZE)


def _fused_agg_output_size(n: int) -> int:
    """Output size for fused_group_agg: num_warps × 6 aggs."""
    from math import ceil
    WARP_SIZE    = 32
    NUM_AGGS     = 6
    return ceil(n / WARP_SIZE) * NUM_AGGS

### ⚡ `gpu_masked_sum_kernel` — warp-reduction masked sum

**Kernel contract** (`masked_sum_simple`):

| Tensor | dtype | shape | description |
|---|---|---|---|
| `mask` | float32 | `[N]` | 1.0 = include row, 0.0 = exclude |
| `values` | float32 | `[N]` | values to sum |
| output | float32 | `[⌈N/32⌉]` | one partial sum per warp |

The Mojo kernel writes **partial sums** (one per warp of 32 threads).  We then wire a final
`ops.sum(partial_sums)` inside the same `Graph` — so the whole chain compiles into
**one fused executable**: single H2D copy → kernel → D2H scalar.

This is the **generic GPU masked-sum** used for arbitrary single-column aggregations with a
boolean mask.

In [ ]:
#| export
def gpu_masked_sum_kernel(
    mask:   np.ndarray,    # Boolean/float32 mask (1.0 = include)
    values: np.ndarray,    # Values to sum where mask is true
    device: DeviceType = "cpu",
) -> float:
    """
    Masked sum via custom Mojo warp-reduction kernel.

    Works on **both** `device='cpu'` and `device='gpu'` — the Mojo kernel
    has branches for both targets compiled into `kernels.mojopkg`.

    - **CPU**: kernel writes a single scalar directly to `output[0]`.
    - **GPU**: kernel writes `ceil(N/32)` warp partial sums; a final
      `ops.sum` in the Graph reduces them to a scalar.

    Falls back to `max_masked_sum()` (pure Graph ops) only when
    `kernels.mojopkg` is not found.

    Args:
        mask:   float32 array, 1.0 = include row
        values: float32 array to sum
        device: `'cpu'` or `'gpu'`

    Returns:
        Scalar sum as float.
    """
    if not kernels_available():
        return max_masked_sum(mask, values, device=device)

    sess = MAXSession.get(device)
    n    = len(mask)
    m_t  = sess.to_tensor(np.ascontiguousarray(mask,   dtype=np.float32))
    v_t  = sess.to_tensor(np.ascontiguousarray(values, dtype=np.float32))

    # CPU kernel writes one final scalar; GPU writes one partial per warp
    if device == "gpu":
        out_size = _partial_sum_size(n)
    else:
        out_size = 1

    key = ("masked_sum_kernel", n, device)

    def build():
        in_tt  = TensorType(DType.float32, (n,),        sess.device_ref)
        out_tt = TensorType(DType.float32, (out_size,), sess.device_ref)
        with Graph(
            "masked_sum_kernel",
            input_types=[in_tt, in_tt],
            custom_extensions=[KERNELS_PATH],
        ) as g:
            m, v     = g.inputs
            partials = ops.custom(
                name      = "masked_sum_simple",
                device    = sess.device_ref,
                values    = [m, v],
                out_types = [out_tt],
            )[0]
            # Reduce partial sums → scalar (one warp partial on CPU, many on GPU)
            g.output(ops.sum(partials))
        return g

    model  = sess._get_or_compile(key, build)
    result = model.execute(m_t, v_t)[0]
    return float(sess.from_tensor(result).flat[0])


# Clean device-agnostic alias — preferred name for new code
masked_sum_kernel = gpu_masked_sum_kernel


### 🔥 `gpu_fused_group_agg` — 6 aggregations in one kernel launch

This is the **Q1 fast path**.  A single `fused_group_agg` kernel computes all six TPC-H
aggregations for **one group** in one pass over the data.  Call it once per group.

**Kernel contract** (`fused_group_agg`):

| Input | dtype | shape | description |
|---|---|---|---|
| `mask` | float32 | `[N]` | group membership: 1.0 = this row belongs to the group |
| `qty` | float32 | `[N]` | `l_quantity` |
| `price` | float32 | `[N]` | `l_extendedprice` |
| `disc` | float32 | `[N]` | `l_discount` |
| `disc_price` | float32 | `[N]` | `l_extendedprice × (1 − l_discount)` |
| `charge` | float32 | `[N]` | `disc_price × (1 + l_tax)` |
| output | float32 | `[⌈N/32⌉ × 6]` | 6 partial sums per warp |

Each warp writes 6 values: `[sum_qty, sum_price, sum_disc, sum_disc_price, sum_charge, count]`.
The Graph reduces them: `ops.sum` on columns 0..5 of the reshaped output.

**Why this is fast**:  
`N × 6 memory reads` in one kernel vs `6 × N reads` with 6 separate masked_sum calls.
On 1M rows ≈ **6× memory bandwidth reduction** per group.

In [ ]:
#| export
# Canonical column order that fused_group_agg kernel expects / returns
_FUSED_AGG_COLS = ("sum_qty", "sum_base_price", "sum_disc", "sum_disc_price", "sum_charge", "count_order")
_FUSED_AGG_IDX  = {name: i for i, name in enumerate(_FUSED_AGG_COLS)}


def gpu_fused_group_agg(
    mask:       np.ndarray,   # float32, shape [N] — group membership mask
    qty:        np.ndarray,   # float32, shape [N]
    price:      np.ndarray,   # float32, shape [N]
    disc:       np.ndarray,   # float32, shape [N]
    disc_price: np.ndarray,   # float32, shape [N]  = price × (1−disc)
    charge:     np.ndarray,   # float32, shape [N]  = disc_price × (1+tax)
    device: DeviceType = "cpu",
) -> dict:
    """
    6-column fused aggregation for one group via custom Mojo kernel.

    Works on **both** `device='cpu'` and `device='gpu'`:

    - **CPU**: Mojo kernel loops over N, accumulates 6 sums directly, writes
      `[6]` output. One kernel call, single O(N) pass over all 6 columns.
    - **GPU**: Mojo kernel uses warp reduction, writes `[ceil(N/32) × 6]`
      warp partials. MAX Graph reduces via reshape/transpose/sum.

    Both paths use one kernel call for 6 aggregations — vs 6 × `max_masked_sum`
    calls (which each do a separate O(N) pass).

    Returns a dict with keys matching `_FUSED_AGG_COLS`:
      `sum_qty`, `sum_base_price`, `sum_disc`,
      `sum_disc_price`, `sum_charge`, `count_order`

    Falls back to six separate `max_masked_sum()` calls only when
    `kernels.mojopkg` is not found.

    Args:
        mask:       float32 boolean mask (1.0 = row in this group)
        qty / price / disc / disc_price / charge: pre-computed columns
        device:     `'cpu'` or `'gpu'`

    Returns:
        dict[str, float] with the 6 aggregation values.
    """
    if not kernels_available():
        ms = lambda v: max_masked_sum(mask, v, device=device)
        return {
            "sum_qty":        ms(qty),
            "sum_base_price": ms(price),
            "sum_disc":       ms(disc),
            "sum_disc_price": ms(disc_price),
            "sum_charge":     ms(charge),
            "count_order":    float(mask.astype(bool).sum()),
        }

    sess = MAXSession.get(device)
    n    = len(mask)

    def _t(arr): return sess.to_tensor(np.ascontiguousarray(arr, dtype=np.float32))
    m_t, q_t, p_t, d_t, dp_t, ch_t = map(_t, [mask, qty, price, disc, disc_price, charge])

    # CPU kernel writes [6] final sums directly (num_warps = 1 logical)
    # GPU kernel writes [num_warps × 6] warp partial sums
    if device == "gpu":
        num_warps = _partial_sum_size(n)
    else:
        num_warps = 1
    out_size = num_warps * 6
    key      = ("fused_group_agg", n, device)

    def build():
        tt     = TensorType(DType.float32, (n,),        sess.device_ref)
        out_tt = TensorType(DType.float32, (out_size,), sess.device_ref)
        with Graph(
            "fused_group_agg",
            input_types=[tt, tt, tt, tt, tt, tt],
            custom_extensions=[KERNELS_PATH],
        ) as g:
            m, q, p, d, dp, ch = g.inputs
            partials = ops.custom(
                name      = "fused_group_agg",
                device    = sess.device_ref,
                values    = [m, q, p, d, dp, ch],
                out_types = [out_tt],
            )[0]
            # Reshape warp partials and reduce each of the 6 agg columns.
            # CPU: [1×6] → [6,1] → sum → [6,1] → [6]  (trivial, no real reduction)
            # GPU: [NW×6] → [6,NW] → sum → [6,1] → [6]
            shaped   = ops.reshape(partials, [num_warps, 6])  # [NW, 6]
            transpos = ops.transpose(shaped, 0, 1)            # [6, NW]
            col_sums = ops.sum(transpos)                      # [6, 1]
            flat6    = ops.reshape(col_sums, [6])             # [6]
            cols     = ops.split(flat6, [1,1,1,1,1,1], 0)    # 6 × [1]
            g.output(*cols)
        return g

    model   = sess._get_or_compile(key, build)
    outputs = model.execute(m_t, q_t, p_t, d_t, dp_t, ch_t)
    return {
        name: float(sess.from_tensor(outputs[i]).flat[0])
        for i, name in enumerate(_FUSED_AGG_COLS)
    }


# Clean device-agnostic alias — preferred name for new code
fused_agg_kernel = gpu_fused_group_agg


### 🏎️ `group_sum_kernel` — single-pass scatter-sum for all groups

This is the key primitive that replaces the `O(groups × n)` masked-sum loop
with an `O(n)` scatter pass across **all groups at once**.

**How it works:**
```
group_ids [N]  values [N]
     │               │
     └────────────── Mojo group_sum kernel ──── single pass O(N)
                     Each thread: local_sums[group_ids[i]] = values[i]
                     Warp reduction → partial sums per group
                     │
                     ▼
     [num_warps × n_groups] warp partials
                     │
     reshape → [NW, G], transpose → [G, NW], sum(-1) → [G, 1], reshape → [G]
                     │
                     ▼
              result [n_groups]    ← all group sums in one kernel call
```

**vs the old masked-sum approach:**  
Old: `48 kernel calls` = 6 groups × 8 agg cols, each doing `O(N)` work  
New: `8 kernel calls` = 1 per agg col, each doing `O(N)` work once for **all groups**

**n_groups is encoded in the output tensor shape** (baked at MAX graph compile time) — no separate scalar input needed. CPU path writes `[n_groups]` directly; GPU path writes `[ceil(N/32) × n_groups]` warp partials then reduces.

In [ ]:
#| export
def group_sum_kernel(
    values:    np.ndarray,    # float32 [N] — values to sum
    group_ids: np.ndarray,    # int32   [N] — 0-based group index per row
    n_groups:  int,           # number of groups (≤ MAX_GROUPS = 64)
    device: DeviceType = "cpu",
) -> np.ndarray:
    """
    Single-pass scatter-sum for ALL groups in one Mojo kernel call.

    Works on **both** `device='cpu'` and `device='gpu'`:

    - **CPU**: Mojo kernel loops over N, accumulates into `[n_groups]`
      output directly. One kernel call, O(N).
    - **GPU**: Mojo kernel uses per-thread local accumulators + warp
      shuffle reduction. Writes `[ceil(N/32) × n_groups]` warp partials;
      MAX Graph reduces via reshape/transpose/sum.

    Computes `result[g] = sum(values[i] for i where group_ids[i] == g)`
    for every group simultaneously — O(N) vs the old O(groups × N) loop.

    **n_groups is baked into the compiled graph output shape.** A new graph
    is compiled (and cached) for each distinct `(N, n_groups, device)` triple.

    Args:
        values:    float32 values to sum, shape [N]
        group_ids: int32 group assignment per row, shape [N], values in [0, n_groups)
        n_groups:  total number of groups; must be ≤ 64 (MAX_GROUPS)
        device:    `'cpu'` or `'gpu'`

    Returns:
        float32 numpy array, shape [n_groups], result[g] = sum for group g.

    Raises:
        RuntimeError: if kernels.mojopkg is not found.
    """
    if not kernels_available():
        raise RuntimeError(
            "group_sum_kernel requires kernels.mojopkg — "
            "run: mojo package archive/kernels -o mxframe/kernels.mojopkg"
        )
    if n_groups > 64:
        raise ValueError(f"n_groups={n_groups} exceeds MAX_GROUPS=64")

    sess = MAXSession.get(device)
    n    = len(values)

    # ── Build tensors ────────────────────────────────────────────────────────
    v_f   = np.ascontiguousarray(values,    dtype=np.float32)
    g_i   = np.ascontiguousarray(group_ids, dtype=np.int32)
    v_t   = sess.to_tensor(v_f)
    g_raw = driver.Tensor.from_numpy(g_i)
    g_t   = g_raw.to(sess.device) if sess.device_type == "gpu" else g_raw

    # ── Output size ──────────────────────────────────────────────────────────
    # CPU kernel writes [n_groups] final sums directly.
    # GPU kernel writes [ceil(N/32) × n_groups] warp partial sums.
    if device == "gpu":
        num_warps = _partial_sum_size(n)
        out_size  = num_warps * n_groups
    else:
        num_warps = 1
        out_size  = n_groups

    key = ("group_sum", n, n_groups, device)

    def build():
        val_tt = TensorType(DType.float32, (n,),        sess.device_ref)
        gid_tt = TensorType(DType.int32,   (n,),        sess.device_ref)
        out_tt = TensorType(DType.float32, (out_size,), sess.device_ref)
        with Graph(
            "group_sum",
            input_types=[val_tt, gid_tt],
            custom_extensions=[KERNELS_PATH],
        ) as g:
            vals_in, gids_in = g.inputs
            partials = ops.custom(
                name      = "group_sum",
                device    = sess.device_ref,
                values    = [vals_in, gids_in],
                out_types = [out_tt],
            )[0]
            # Reduce warp partials → [n_groups]
            # CPU (num_warps=1): [1,G] → [G,1] → sum → [G,1] → [G]  (trivial)
            # GPU (num_warps=NW): [NW,G] → [G,NW] → sum → [G,1] → [G]
            shaped   = ops.reshape(partials, [num_warps, n_groups])
            transpos = ops.transpose(shaped, 0, 1)           # [n_groups, num_warps]
            col_sums = ops.sum(transpos)                     # [n_groups, 1]
            flat     = ops.reshape(col_sums, [n_groups])     # [n_groups]
            g.output(flat)
        return g

    model  = sess._get_or_compile(key, build)
    result = model.execute(v_t, g_t)[0]
    return sess.from_tensor(result).flatten().astype(np.float32)


## Tests

In [ ]:
import numpy as np, time
from max import driver

GPU_AVAILABLE = driver.accelerator_count() > 0
DEVICES       = ["cpu"] + (["gpu"] if GPU_AVAILABLE else [])

print(f"GPU available     : {GPU_AVAILABLE}")
print(f"kernels_available : {kernels_available()}")
print(f"KERNELS_PATH      : {KERNELS_PATH}")
print(f"Devices under test: {DEVICES}")
print()

# ── Test data ────────────────────────────────────────────────────────────────
N    = 100_000
rng  = np.random.default_rng(42)
arr  = rng.uniform(0, 1, N).astype(np.float32)
mask = (rng.random(N) > 0.5).astype(np.float32)  # ~50% ones
expected_sum  = float(arr.sum())
expected_msum = float((mask * arr).sum())

# ── Helper: run + time a function ───────────────────────────────────────────
def bench(fn, *args, warmup=1, runs=3, **kw):
    for _ in range(warmup): fn(*args, **kw)
    t0 = time.perf_counter()
    for _ in range(runs):   fn(*args, **kw)
    return (time.perf_counter() - t0) / runs * 1000  # ms per call

# ─────────────────────────────────────────────────────────────────────────────
# 1. max_sum  —  same call, same result, different device
# ─────────────────────────────────────────────────────────────────────────────
print("── max_sum ─────────────────────────────────────────────────")
for dev in DEVICES:
    s = max_sum(arr, device=dev)
    assert abs(s - expected_sum) < 1.0, f"[{dev}] got {s:.2f}, expected {expected_sum:.2f}"
    ms = bench(max_sum, arr, device=dev)
    print(f"  max_sum(arr, device='{dev}')  →  {s:.2f}   [{ms:.2f} ms/call]")

# ─────────────────────────────────────────────────────────────────────────────
# 2. max_masked_sum  —  Graph-op path, device-transparent
# ─────────────────────────────────────────────────────────────────────────────
print("\n── max_masked_sum ──────────────────────────────────────────")
for dev in DEVICES:
    s = max_masked_sum(mask, arr, device=dev)
    assert abs(s - expected_msum) < 1.0, f"[{dev}] got {s:.2f}, expected {expected_msum:.2f}"
    ms = bench(max_masked_sum, mask, arr, device=dev)
    print(f"  max_masked_sum(mask, arr, device='{dev}')  →  {s:.2f}   [{ms:.2f} ms/call]")

# ─────────────────────────────────────────────────────────────────────────────
# 3. masked_sum_kernel  —  Mojo kernel path (cpu branch or gpu branch)
# ─────────────────────────────────────────────────────────────────────────────
print("\n── masked_sum_kernel  (Mojo kernel on both devices) ────────")
for dev in DEVICES:
    s = masked_sum_kernel(mask, arr, device=dev)
    assert abs(s - expected_msum) < 1.0, f"[{dev}] got {s:.2f}, expected {expected_msum:.2f}"
    ms = bench(masked_sum_kernel, mask, arr, device=dev)
    print(f"  masked_sum_kernel(mask, arr, device='{dev}')  →  {s:.2f}   [{ms:.2f} ms/call]")

# ─────────────────────────────────────────────────────────────────────────────
# 4. fused_agg_kernel  —  6 aggregations, one kernel call
# ─────────────────────────────────────────────────────────────────────────────
print("\n── fused_agg_kernel  (6 aggs, 1 kernel call) ──────────────")
qty   = arr.copy()
price = (arr * 100).astype(np.float32)
disc  = (rng.uniform(0, 0.1, N)).astype(np.float32)
dp    = (price * (1 - disc)).astype(np.float32)
chg   = (dp * 1.04).astype(np.float32)

for dev in DEVICES:
    res = fused_agg_kernel(mask, qty, price, disc, dp, chg, device=dev)
    ms  = bench(fused_agg_kernel, mask, qty, price, disc, dp, chg, device=dev)
    print(f"  fused_agg_kernel(... device='{dev}')  [{ms:.2f} ms/call]")
    print(f"    sum_qty={res['sum_qty']:.0f}  count={res['count_order']:.0f}  sum_disc_price={res['sum_disc_price']:.0f}")

# ─────────────────────────────────────────────────────────────────────────────
# 5. group_sum_kernel  —  all groups in one scatter pass
# ─────────────────────────────────────────────────────────────────────────────
print("\n── group_sum_kernel  (scatter-sum, all groups at once) ─────")
n_grps = 6
gids   = (rng.integers(0, n_grps, N)).astype(np.int32)
vals   = arr.copy()
# Reference sums
ref = np.array([float(vals[gids == g].sum()) for g in range(n_grps)], dtype=np.float32)

for dev in DEVICES:
    r  = group_sum_kernel(vals, gids, n_grps, device=dev)
    ms = bench(group_sum_kernel, vals, gids, n_grps, device=dev)
    max_err = float(np.abs(r - ref).max())
    assert max_err < 0.5, f"[{dev}] max error={max_err:.4f}"
    print(f"  group_sum_kernel(vals, gids, n_grps={n_grps}, device='{dev}')  max_err={max_err:.4f}   [{ms:.2f} ms/call]")
    print(f"    per-group sums: {r.round(1).tolist()}")

print("\n✅ All MAX Engine tests passed — CPU and GPU API are symmetric")
